# Nettoyage & enrichissement des boxscores (minutes, DNP, totaux, ratios)

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from nba_api.stats.static import players

from src.config import *
from src.utils import *
from src.feature_builder import *
from src.feature_aggregation import *

#display full columns
pd.set_option('display.max_column', None)
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_seq_items', None)
# pd.set_option('display.max_colwidth', 500)
# pd.set_option('expand_frame_repr', True)

In [2]:
#store start time of notebook
start_time = datetime.now()
print("Start time: ", start_time)

Start time:  2025-06-09 00:43:08.365194


# 🔁 Chargement des fichiers

In [3]:
boxscores_file = get_latest_file(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR)
games_file = get_latest_file(DATA_LAST_GAMES_MERGED_DIR)
df_boxscores = pd.read_csv(boxscores_file, dtype={'gameId': str})
df_games = pd.read_csv(games_file, dtype={'GAME_ID': str})


In [4]:
games_file

'data\\raw_last\\games_merged\\games_merged_all_seasons__2025-06-06_19-44-58.csv'

In [5]:
df_boxscores

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage
0,0021000002,1610612756,Phoenix,Suns,PHX,suns,255,Grant,Hill,G. Hill,grant-hill,F,NaN,NaN,26:05,106.3,122.9,-8.9,-16.7,0.056,0.33,12.5,0.050,0.000,0.021,37.5,0.500,0.450,0.130,89.40,88.33,48.0,-0.011,0.605,0.237,0.234,0.529,0.098,0.199,0.591,0,0,0,4,12,13,10,24,0,3,0.750,0.250,1.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,1.000,0.000,0.000,0.000,1.000,2,4,0,1,0.00,0,1,0.0,1,0,1,1,0,1,3,1,4,-8.0,0.100,0.105,0.000,0.091,0.000,0.111,0.167,0.000,0.067,0.143,0.273,0.000,1.0,0.0,0.111,0.273,0.078
1,0021000002,1610612756,Phoenix,Suns,PHX,suns,2045,Hedo,Turkoglu,H. Turkoglu,hedo-turkoglu,F,NaN,NaN,27:13,114.9,124.5,-11.6,-9.6,0.100,2.00,20.0,0.000,0.111,0.060,10.0,0.429,0.429,0.145,86.77,84.65,47.0,0.010,0.556,0.133,0.164,0.548,0.077,0.161,0.522,0,0,0,0,6,15,0,22,0,1,0.429,0.571,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.500,0.500,0.500,0.500,2,7,2,4,0.50,0,0,0.0,0,3,3,2,1,0,1,5,6,-7.0,0.091,0.156,0.333,0.267,0.000,0.000,0.000,0.273,0.167,0.286,0.125,0.333,0.0,0.0,0.556,0.125,0.111
2,0021000002,1610612756,Phoenix,Suns,PHX,suns,201577,Robin,Lopez,R. Lopez,robin-lopez,C,NaN,NaN,24:50,120.0,122.7,6.2,-2.7,0.000,0.00,0.0,0.158,0.111,0.130,16.7,0.500,0.512,0.118,91.52,86.00,45.0,0.071,0.641,0.179,0.195,0.490,0.122,0.226,0.667,1,4,0,4,10,18,10,24,0,2,1.000,0.000,0.800,0.000,0.000,0.000,0.200,0.200,0.800,0.000,1.000,0.000,0.000,0.000,1.000,2,4,0,0,0.00,1,2,0.5,3,3,6,0,0,1,1,0,5,0.0,0.091,0.

In [6]:
df_games

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
0,22010,1610612756,PHX,Phoenix Suns,0021000002,2010-10-26,PHX @ POR,L,239,92,36,74,0.486,9,19,0.474,11,16,0.688,7,23,30,15,3,4,19,19,-14.0,2010-11
1,22010,1610612747,LAL,Los Angeles Lakers,0021000003,2010-10-26,LAL vs. HOU,W,240,112,40,96,0.417,9,21,0.429,23,28,0.821,14,30,44,21,11,4,12,24,2.0,2010-11
2,22010,1610612748,MIA,Miami Heat,0021000001,2010-10-26,MIA @ BOS,L,242,80,27,74,0.365,8,20,0.400,18,25,0.720,11,28,39,15,10,6,17,21,-8.0,2010-11
3,22010,1610612745,HOU,Houston Rockets,0021000003,2010-10-26,HOU @ LAL,L,240,110,38,91,0.418,8,20,0.400,26,28,0.929,16,37,53,25,6,7,20,25,-2.0,2010-11
4,22010,1610612738,BOS,Boston Celtics,0021000001,2010-10-26,BOS vs. MIA,W,239,88,32,69,0.464,8,16,0.500,16,25,0.640,8,34,42,25,6,4,18,19,8.0,2010-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48501,42024,1610612752,NYK,New York Knicks,0042400305,2025-05-29,NYK vs. IND,W,241,111,44,89,0.494,8,29,0.276,15,22,0.682,11,34,45,22,11,3,15,22,17.0,2024-25
48502,42024,1610612754,IND,Indiana Pacers,0042400306,2025-05-31,IND vs. NYK,W,241,125,46,85,0.541,17,33,0.515,16,19,0.842,6,30,36,30,10,9,12,23,17.0,2024-25
48503,42024,1610612752,NYK,New York Knicks,0042400306,2025-05-31,NYK @ IND,L,240,108,41,86,0.477,9,32,0.281,17,26,0.654,13,28,41,23,10,6,17,16,-17.0,2024-25
48504,42024,1610612754,IND,Indiana Pacers,0042400401,2025-06-05,IND @ OKC,W,240,111,39,82,0.476,18,39,0.462,15,21,0.714,13,43,56,24,1,7,24,22,1.0,2024-25


# 🧼 Nettoyage des minutes jouées

In [7]:
def convert_minutes(val):
    if pd.isna(val) or val in ['DNP', '']:
        return 0.0
    try:
        parts = str(val).split(':')
        return int(parts[0]) + int(parts[1]) / 60 if len(parts) == 2 else float(val)
    except:
        return 0.0

df_boxscores['MINUTES_PLAYED'] = df_boxscores['minutes'].apply(convert_minutes)

# Rename gameId and teamId in boxscores

In [8]:
rename_columns = {
    'gameId': 'GAME_ID',
    'teamId': 'TEAM_ID',
}

df_boxscores.rename(columns=rename_columns, inplace=True)


# 🔄 Cast dynamique des colonnes numériques


In [9]:
numeric_cols = df_boxscores.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    df_boxscores[col] = pd.to_numeric(df_boxscores[col], errors='coerce').fillna(0)

In [10]:
df_boxscores

,GAME_ID,TEAM_ID,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,playerSlug,position,comment,jerseyNum,minutes,offensiveRating_advanced,defensiveRating_advanced,estimatedNetRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,oppPointsOffTurnovers_misc,oppPointsSecondChance_misc,oppPointsFastBreak_misc,oppPointsPaint_misc,blocksAgainst_misc,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,threePointersPercentage_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,freeThrowsPercentage_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,plusMinusPoints_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,MINUTES_PLAYED
0,0021000002,1610612756,Phoenix,Suns,PHX,suns,255,Grant,Hill,G. Hill,grant-hill,F,NaN,0.0,26:05,106.3,122.9,-8.9,-16.7,0.056,0.33,12.5,0.050,0.000,0.021,37.5,0.500,0.450,0.130,89.40,88.33,48.0,-0.011,0.605,0.237,0.234,0.529,0.098,0.199,0.591,0,0,0,4,12,13,10,24,0,3,0.750,0.250,1.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,1.000,0.000,0.000,0.000,1.000,2,4,0,1,0.00,0,1,0.0,1,0,1,1,0,1,3,1,4,-8.0,0.100,0.105,0.000,0.091,0.000,0.111,0.167,0.000,0.067,0.143,0.273,0.000,1.0,0.0,0.111,0.273,0.078,26.083333
1,0021000002,1610612756,Phoenix,Suns,PHX,suns,2045,Hedo,Turkoglu,H. Turkoglu,hedo-turkoglu,F,NaN,0.0,27:13,114.9,124.5,-11.6,-9.6,0.100,2.00,20.0,0.000,0.111,0.060,10.0,0.429,0.429,0.145,86.77,84.65,47.0,0.010,0.556,0.133,0.164,0.548,0.077,0.161,0.522,0,0,0,0,6,15,0,22,0,1,0.429,0.571,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.500,0.500,0.500,0.500,2,7,2,4,0.50,0,0,0.0,0,3,3,2,1,0,1,5,6,-7.0,0.091,0.156,0.333,0.267,0.000,0.000,0.000,0.273,0.167,0.286,0.125,0.333,0.0,0.0,0.556,0.125,0.111,27.216667
2,0021000002,1610612756,Phoenix,Suns,PHX,suns,201577,Robin,Lopez,R. Lopez,robin-lopez,C,NaN,0.0,24:50,120.0,122.7,6.2,-2.7,0.000,0.00,0.0,0.158,0.111,0.130,16.7,0.500,0.512,0.118,91.52,86.00,45.0,0.071,0.641,0.179,0.195,0.490,0.122,0.226,0.667,1,4,0,4,10,18,10,24,0,2,1.000,0.000,0.800,0.000,0.000,0.000,0.200,0.200,0.800,0.000,1.000,0.000,0.000,0.000,1.000,2,4,0,0,0.00,1

# Clean duplicates boxscores

In [11]:
# remove duplicate rows based on 'GAME_ID' and 'TEAM_ID' and "playerSlug"
df_boxscores = df_boxscores.drop_duplicates(subset=['GAME_ID', 'TEAM_ID', 'playerSlug'])

# Build player features

In [12]:
player_features_df = build_player_status_features(df_boxscores)

today = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')


os.makedirs(DATA_PLAYERS_DIR, exist_ok=True)
player_features_output = os.path.join(DATA_PLAYERS_DIR, f'player_features_{today}.csv')

# to csv
#player_features_df.to_csv(player_features_output, index=False)

In [13]:
player_features_df

,GAME_ID,TEAM_ID,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment
0,0021000002,1610612756,255,1,0,0,0,0,0,4.7,0.5,3.3,-0.011,
1,0021000002,1610612756,2045,1,0,0,0,0,0,12.6,-3.5,9.2,0.010,
2,0021000002,1610612756,201577,1,0,0,0,0,0,12.2,1.5,5.0,0.071,
3,0021000002,1610612756,2202,1,0,0,0,0,0,30.7,1.5,26.9,0.184,
4,0021000002,1610612756,959,1,0,0,0,0,0,30.6,0.5,31.2,0.127,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948269,0042400401,1610612760,1629026,1,0,0,0,0,0,0.0,0.0,0.0,0.000,
948270,0042400401,1610612760,1642349,1,0,0,0,0,0,3.6,0.0,0.0,-0.083,
948271,0042400401,1610612760,1631172,0,1,0,0,0,0,0.0,0.0,0.0,0.000,dnp - coach's decision
948272,0042400401,1610612760,1641794,0,1,0,0,0,0,0.0,0.0,0.0,0.000,dnp - coach's decision


In [14]:
#player_features_df[player_features_df['is_absent'] == 1].sort_values(by='GAME_ID').head(50)

#same but filter with comment "DNP - Coach's Decision"

filtered = player_features_df[
    (player_features_df['is_absent'] == 1) &
    (~player_features_df['comment'].str.contains("coach's decision", na=False))
]
display(filtered.sort_values(by=['GAME_ID', 'TEAM_ID']).tail(50))

filtered = player_features_df[
    (player_features_df['is_personal'] == 1) &
    (~player_features_df['comment'].str.contains("personal", na=False)) &
    (~player_features_df['comment'].str.contains("not with team", na=False)) 
]
display(filtered.sort_values(by=['GAME_ID', 'TEAM_ID']).tail(50))




# display(player_features_df[
#     player_features_df['is_personal'] == 1 &
#     (~player_features_df['comment'].str.contains("nwt", na=False))
#     ].head(50))


,GAME_ID,TEAM_ID,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment
514612,0021801052,1610612743,1629020,0,1,0,0,0,0,0.0,0.0,0.0,0.0,
517602,0021801174,1610612756,2037,0,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd_coach
517826,0021801187,1610612742,1628961,0,1,0,0,0,0,0.0,0.0,0.0,0.0,dnd_coach
563883,0021900398,1610612742,1626246,0,1,0,0,0,0,0.0,0.0,0.0,0.0,
565484,0021900464,1610612758,1629713,0,1,0,0,0,0,-1.0,0.0,-1.0,1.0,
573084,0021900770,1610612742,1626246,0,1,0,0,0,0,0.0,0.0,0.0,0.0,
579425,0021901281,1610612743,1629626,0,1,0,0,0,0,0.0,0.0,0.0,0.0,
579413,0021901281,1610612762,1629730,0,1,0,0,0,0,0.0,0.0,0.0,0.0,
617776,0022000485,1610612738,1629605,0,1,0,0,0,0,0.0,0.0,0.0,0.0,
623462,0022000678,1610612748,202340,0,1,0,0,0,0,0.0,0.0,0.0,0.0,nwt - trade pending


,GAME_ID,TEAM_ID,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment
16303,0021000686,1610612765,1112,0,0,0,0,0,1,0.0,0.0,0.0,0.0,nwt - family matter
16830,0021000705,1610612765,1112,0,0,0,0,0,1,0.0,0.0,0.0,0.0,nwt - family matter
20912,0021000874,1610612765,1112,0,0,0,0,0,1,0.0,0.0,0.0,0.0,nwt - family matter
21593,0021000906,1610612750,2545,0,0,0,0,0,1,0.0,0.0,0.0,0.0,nwt - family matter
26648,0021001107,1610612765,101198,0,0,0,0,0,1,0.0,0.0,0.0,0.0,nwt - family matter
53094,0021100099,1610612765,2732,0,0,0,0,0,1,0.0,0.0,0.0,0.0,nwt - family matters
57762,0021100181,1610612765,2863,0,0,0,0,0,1,0.0,0.0,0.0,0.0,nwt - family matter
33188,0021100260,1610612765,2419,0,0,0,0,0,1,0.0,0.0,0.0,0.0,dnd - family matter
33266,0021100274,1610612765,2419,0,0,0,0,0,1,0.0,0.0,0.0,0.0,nwt - family matter
46386,0021100741,1610612757,101109,0,0,0,0,0,1,0.0,0.0,0.0,0.0,nwt - family


In [15]:

# Analyse des statuts des joueurs
total_rows = len(player_features_df)

# Comptage des cas
n_present = player_features_df['is_present'].sum()
n_absent = player_features_df['is_absent'].sum()
n_injured = player_features_df['is_injured'].sum()

# is_resting
# is_suspended
# is_personal

n_resting = player_features_df['is_resting'].sum()
n_suspended = player_features_df['is_suspended'].sum()
n_personal = player_features_df['is_personal'].sum()

# Lignes incohérentes : aucune des colonnes n'est True

mask_no_status = (
    (player_features_df['is_present'] == 0) &
    (player_features_df['is_absent'] == 0) &
    (player_features_df['is_injured'] == 0) &
    (player_features_df['is_resting'] == 0) &
    (player_features_df['is_suspended'] == 0) &
    (player_features_df['is_personal'] == 0)
)


n_inconsistent = mask_no_status.sum()

# Lignes avec plusieurs statuts à la fois (logiquement impossible)
mask_multiple_status = (
    player_features_df[['is_present', 'is_absent', 'is_injured']].sum(axis=1) > 1
)
n_multiple = mask_multiple_status.sum()

print(f"✅ Analyse des statuts des joueurs sur {total_rows} lignes")
print(f" - Joueurs présents : {n_present}")
print(f" - Joueurs absents : {n_absent}")
print(f" - Joueurs blessés : {n_injured}")
print(f" - Joueurs en repos : {n_resting}")
print(f" - Joueurs suspendus : {n_suspended}")
print(f" - Joueurs pour raisons personnelles : {n_personal}")
print(f"❌ Lignes sans statut défini : {n_inconsistent}")
print(f"⚠️ Lignes avec plusieurs statuts actifs : {n_multiple}")

display(player_features_df[mask_no_status].head(10))



✅ Analyse des statuts des joueurs sur 489857 lignes
 - Joueurs présents : 403786
 - Joueurs absents : 72372
 - Joueurs blessés : 12267
 - Joueurs en repos : 379
 - Joueurs suspendus : 482
 - Joueurs pour raisons personnelles : 571
❌ Lignes sans statut défini : 0
⚠️ Lignes avec plusieurs statuts actifs : 0


,GAME_ID,TEAM_ID,personId,is_present,is_absent,is_injured,is_resting,is_suspended,is_personal,player_perf_score,player_defense_score,player_offense_score,player_impact_score,comment


# ⚙️ Aggrégation des data Player par équipe et match


In [16]:
# df_team_players = aggregate_team_player_features(player_features_df, top_n=10)

# display(df_team_players.head(10))
# display(df_team_players.tail(10))

# #display some lines with absent players
# absent_players = df_team_players[df_team_players['num_injured'] >= 1]
# absent_players 

# Identification des top joueurs sur l'ensemble du dataset
df_top_players = identify_historical_top_players(player_features_df)
print(f"✅ {df_top_players['tag_top_player'].sum()} top joueurs identifiés sur {len(df_top_players)} joueurs.")


df_agg_actual = aggregate_actual_team_features(player_features_df)
print(f"✅ {df_agg_actual.shape[0]} lignes générées dans l'aggregation actuelle par équipe.")

df_agg_top_abs = flag_top_players_absences(player_features_df, df_top_players)
print(f"✅ {df_agg_top_abs['top_player_absence_count'].sum()} absences de top joueurs détectées.")


df_team_features_final = df_agg_actual.merge(df_agg_top_abs, on=["GAME_ID", "TEAM_ID"], how="left")
df_team_features_final.fillna(0, inplace=True)
print(f"✅ Fusion réussie. Shape finale : {df_team_features_final.shape}")


KeyError: "['GAME_DATE'] not in index"

In [ ]:
display(df_team_features_final.head(10))
display(df_team_features_final.tail(10))

display(df_team_features_final[df_team_features_final['top_player_absence_count'] > 0].head(20))


,GAME_ID,TEAM_ID,team_perf_score_sum,team_defense_score_sum,team_offense_score_sum,team_impact_score_sum,team_num_players_present,team_num_players_absent,team_num_injured,team_num_resting,team_num_suspended,team_num_personal,team_presence_rate,top_player_absence_count,top_injured,top_resting,top_suspended,top_personal
0,0021000001,1610612738,167.9,-4.0,125.1,0.860,9,2,0,0,1,0,0.818182,2.0,0.0,0.0,0.0,0.0
1,0021000001,1610612748,148.3,3.0,101.5,0.806,9,3,0,0,0,0,0.750000,3.0,0.0,0.0,0.0,0.0
2,0021000002,1610612756,138.5,-8.5,113.9,0.645,10,2,0,0,0,0,0.833333,2.0,0.0,0.0,0.0,0.0
3,0021000002,1610612757,211.1,-2.5,161.7,-4.608,10,2,0,0,0,0,0.833333,2.0,0.0,0.0,0.0,0.0
4,0021000003,1610612745,204.1,-5.5,149.5,0.831,10,2,0,0,0,0,0.833333,2.0,0.0,0.0,0.0,0.0
5,0021000003,1610612747,199.3,-1.5,153.9,0.547,10,2,0,0,0,0,0.833333,2.0,0.0,0.0,0.0,0.0
6,0021000004,1610612738,158.6,-10.5,121.4,-0.574,10,1,0,0,1,0,0.909091,1.0,0.0,0.0,0.0,0.0
7,0021000004,1610612739,170.4,-6.5,136.0,0.922,9,3,0,0,0,0,0.750000,3.0,0.0,0.0,0.0,0.0
8,0021000005,1610612751,176.3,-11.0,135.7,0.479,12,0,0,0,0,0,1.000000,0.0,0.0,0.0,0.0,0.0
9,0021000005,1610612765,177.9,-4.0,143.1,0.936,10,2,0,0,0,0,0.833333,2.0,0.0,0.0,0.0,0.0


,GAME_ID,TEAM_ID,team_perf_score_sum,team_defense_score_sum,team_offense_score_sum,team_impact_score_sum,team_num_players_present,team_num_players_absent,team_num_injured,team_num_resting,team_num_suspended,team_num_personal,team_presence_rate,top_player_absence_count,top_injured,top_resting,top_suspended,top_personal
38296,0052400111,1610612741,154.1,2.5,113.5,0.814,11,1,0,0,0,0,0.916667,1.0,0.0,0.0,0.0,0.0
38297,0052400111,1610612748,210.0,9.5,160.8,0.059,11,2,2,0,0,0,0.846154,2.0,0.0,0.0,0.0,0.0
38298,0052400121,1610612744,217.3,1.0,178.7,0.831,10,5,0,0,0,0,0.666667,5.0,0.0,0.0,0.0,0.0
38299,0052400121,1610612763,195.0,-18.5,153.2,0.573,9,4,0,0,0,0,0.692308,4.0,0.0,0.0,0.0,0.0
38300,0052400131,1610612742,220.7,10.5,178.5,0.545,13,0,0,0,0,0,1.000000,0.0,0.0,0.0,0.0,0.0
38301,0052400131,1610612758,185.3,-4.0,143.7,1.746,12,1,0,0,0,0,0.923077,1.0,0.0,0.0,0.0,0.0
38302,0052400201,1610612737,217.8,4.5,172.8,0.645,9,2,1,0,0,0,0.818182,2.0,0.0,0.0,0.0,0.0
38303,0052400201,1610612748,235.3,3.0,183.1,0.926,9,5,0,0,0,1,0.642857,5.0,0.0,0.0,0.0,0.0
38304,0052400211,1610612742,187.7,8.5,148.7,0.529,11,2,0,0,0,0,0.846154,2.0,0.0,0.0,0.0,0.0
38305,0052400211,1610612763,227.0,1.0,179.0,0.848,12,1,0,0,0,0,0.923077,1.0,0.0,0.0,0.0,0.0


,GAME_ID,TEAM_ID,team_perf_score_sum,team_defense_score_sum,team_offense_score_sum,team_impact_score_sum,team_num_players_present,team_num_players_absent,team_num_injured,team_num_resting,team_num_suspended,team_num_personal,team_presence_rate,top_player_absence_count,top_injured,top_resting,top_suspended,top_personal
0,0021000001,1610612738,167.9,-4.0,125.1,0.860,9,2,0,0,1,0,0.818182,2.0,0.0,0.0,0.0,0.0
1,0021000001,1610612748,148.3,3.0,101.5,0.806,9,3,0,0,0,0,0.750000,3.0,0.0,0.0,0.0,0.0
2,0021000002,1610612756,138.5,-8.5,113.9,0.645,10,2,0,0,0,0,0.833333,2.0,0.0,0.0,0.0,0.0
3,0021000002,1610612757,211.1,-2.5,161.7,-4.608,10,2,0,0,0,0,0.833333,2.0,0.0,0.0,0.0,0.0
4,0021000003,1610612745,204.1,-5.5,149.5,0.831,10,2,0,0,0,0,0.833333,2.0,0.0,0.0,0.0,0.0
5,0021000003,1610612747,199.3,-1.5,153.9,0.547,10,2,0,0,0,0,0.833333,2.0,0.0,0.0,0.0,0.0
6,0021000004,1610612738,158.6,-10.5,121.4,-0.574,10,1,0,0,1,0,0.909091,1.0,0.0,0.0,0.0,0.0
7,0021000004,1610612739,170.4,-6.5,136.0,0.922,9,3,0,0,0,0,0.750000,3.0,0.0,0.0,0.0,0.0
9,0021000005,1610612765,177.9,-4.0,143.1,0.936,10,2,0,0,0,0,0.833333,2.0,0.0,0.0,0.0,0.0
10,0021000006,1610612748,161.9,3.0,122.9,0.804,11,1,0,0,0,0,0.916667,1.0,0.0,0.0,0.0,0.0


# ⚙️ Aggrégation des data Boxscore purs par équipe et match


## Définir les colonnes à sommer et à moyenner pondérées


In [ ]:

cols_to_sum = [
    'fieldGoalsMade_traditional', 'fieldGoalsAttempted_traditional',
    'threePointersMade_traditional', 'threePointersAttempted_traditional',
    'freeThrowsMade_traditional', 'freeThrowsAttempted_traditional',
    'reboundsOffensive_traditional', 'reboundsDefensive_traditional',
    'reboundsTotal_traditional', 'assists_traditional', 'steals_traditional',
    'blocks_traditional', 'turnovers_traditional', 'foulsPersonal_traditional',
    'points_traditional', 'MINUTES_PLAYED',
    'pointsOffTurnovers_misc', 'pointsSecondChance_misc', 'pointsFastBreak_misc',
    'pointsPaint_misc', 'blocksAgainst_misc'
]

cols_to_weighted_avg = [
    'offensiveRating_advanced', 'defensiveRating_advanced', 'netRating_advanced',
    'assistPercentage_advanced', 'assistToTurnover_advanced', 'assistRatio_advanced',
    'offensiveReboundPercentage_advanced', 'defensiveReboundPercentage_advanced',
    'reboundPercentage_advanced', 'turnoverRatio_advanced', 'effectiveFieldGoalPercentage_advanced',
    'trueShootingPercentage_advanced', 'usagePercentage_advanced', 'estimatedPace_advanced',
    'pace_advanced', 'possessions_advanced', 'PIE_advanced',
    'effectiveFieldGoalPercentage_fourfactors', 'freeThrowAttemptRate_fourfactors',
    'teamTurnoverPercentage_fourfactors', 'oppEffectiveFieldGoalPercentage_fourfactors',
    'oppFreeThrowAttemptRate_fourfactors', 'oppTeamTurnoverPercentage_fourfactors',
    'oppOffensiveReboundPercentage_fourfactors', 'foulsDrawn_misc',
    'percentageFieldGoalsAttempted2pt_scoring', 'percentageFieldGoalsAttempted3pt_scoring',
    'percentagePoints2pt_scoring', 'percentagePointsMidrange2pt_scoring',
    'percentagePoints3pt_scoring', 'percentagePointsFastBreak_scoring',
    'percentagePointsFreeThrow_scoring', 'percentagePointsOffTurnovers_scoring',
    'percentagePointsPaint_scoring', 'percentageAssisted2pt_scoring',
    'percentageUnassisted2pt_scoring', 'percentageAssisted3pt_scoring',
    'percentageUnassisted3pt_scoring', 'percentageAssistedFGM_scoring',
    'percentageUnassistedFGM_scoring', 'threePointersPercentage_traditional',
    'freeThrowsPercentage_traditional', 'percentageFieldGoalsMade_usage',
    'percentageFieldGoalsAttempted_usage', 'percentageThreePointersMade_usage',
    'percentageThreePointersAttempted_usage', 'percentageFreeThrowsMade_usage',
    'percentageFreeThrowsAttempted_usage', 'percentageReboundsOffensive_usage',
    'percentageReboundsDefensive_usage', 'percentageReboundsTotal_usage',
    'percentageAssists_usage', 'percentageTurnovers_usage', 'percentageSteals_usage',
    'percentageBlocks_usage', 'percentageBlocksAllowed_usage', 'percentagePersonalFouls_usage',
    'percentagePersonalFoulsDrawn_usage', 'percentagePoints_usage','plusMinusPoints_traditional'
]





## 📊 Agrégation des données par équipe et match


In [ ]:

# 1. Agrégation par somme
group_keys = ['GAME_ID', 'TEAM_ID']
sum_agg = df_boxscores[group_keys + cols_to_sum].copy()
sum_agg = sum_agg.groupby(group_keys).sum().reset_index()

# 2. Agrégation pondérée par les minutes jouées
weighted_agg = compute_weighted_mean_features(df_boxscores, group_keys, cols_to_weighted_avg, weight_col='MINUTES_PLAYED')

# 3. Fusion des deux agrégats
team_match_stats = pd.merge(sum_agg, weighted_agg, on=group_keys, how='left')

# 4. Identifier l'équipe adverse
teams_in_game = df_boxscores.groupby('GAME_ID')['TEAM_ID'].unique().to_dict()
team_match_stats['OPP_TEAM_ID'] = team_match_stats.apply(
    lambda row: [tid for tid in teams_in_game[row['GAME_ID']] if tid != row['TEAM_ID']][0], axis=1
)


# Ajout GAME_DATE


In [ ]:
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])
team_match_stats = team_match_stats.merge(
    df_games[['GAME_ID', 'TEAM_ID', 'GAME_DATE']],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [ ]:

team_match_stats

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,GAME_DATE
0,0021000001,1610612738,32,69,8,16,16,25,8,34,42,25,6,4,18,19,88,240.000000,17,11,11,38,6,95.258826,86.767938,8.458819,0.179324,1.418765,16.786944,0.040284,0.143548,0.097696,13.576424,0.536647,0.565526,0.192004,91.500720,92.499906,61.978819,0.121102,0.524833,0.364811,0.217790,0.420576,0.341330,0.187027,0.244226,2.633472,0.773753,0.226247,0.600667,0.136254,0.202795,0.091713,0.196539,0.204380,0.464413,0.525706,0.421030,0.332569,0.000000,0.710834,0.235902,0.228976,0.522179,0.200810,0.200914,0.180178,0.199018,0.197938,0.195703,0.190636,0.204025,0.202745,0.190902,0.178942,0.176299,0.151945,0.196471,0.197720,0.199016,0.200868,4.585556,1610612748,2010-10-26
1,0021000001,1610612748,27,74,8,20,18,25,11,28,39,15,10,6,17,21,80,240.000000,22,5,11,24,4,86.585597,94.558222,-7.960403,0.158097,0.533849,13.530931,0.049858,0.138836,0.087374,10.813681,0.426550,0.495102,0.197047,91.499007,92.600174,59.272778,0.090339,0.418351,0.344338,0.188392,0.524045,0.364850,0.217761,0.223355,2.708681,0.712399,0.243379,0.496113,0.140206,0.278267,0.123794,0.225773,0.204365,0.355907,0.308280,0.538734,0.396736,0.152847,0.520927,0.434698,0.226023,0.579196,0.205427,0.199612,0.187020,0.186814,0.196853,0.187300,0.218039,0.202114,0.198359,0.228813,0.200531,0.173722,0.193393,0.147483,0.203271,0.192911,0.203349,-6.306528,1610612738,2010-10-26
2,0021000002,1610612756,36,74,9,19,11,16,7,23,30,15,3,4,19,19,92,240.000000,17,11,6,44,2,101.425007,116.062382,-14.648243,0.113202,0.765569,12.112500,0.037518,0.088750,0.065437,15.816979,0.525870,0.535510,0.195844,94.820024,91.299769,49.458125,0.072061,0.551087,0.219725,0.204248,0.516365,0.162388,0.

# 🔁 Merge avec l’adversaire

In [ ]:
team_cols = [col for col in team_match_stats.columns if col not in ['GAME_ID', 'TEAM_ID', 'OPP_TEAM_ID']]
opp_cols = [f"OPP_{col}" for col in team_cols]
df_opp = team_match_stats.rename(columns={col: f"OPP_{col}" for col in team_cols}).rename(
    columns={'TEAM_ID': 'OPP_TEAM_ID', 'OPP_TEAM_ID': 'TEAM_ID'})
match_dataset = pd.merge(
    team_match_stats,
    df_opp[['GAME_ID', 'TEAM_ID'] + opp_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [ ]:
match_dataset.columns

Index(['GAME_ID', 'TEAM_ID', 'fieldGoalsMade_traditional',
       'fieldGoalsAttempted_traditional', 'threePointersMade_traditional',
       'threePointersAttempted_traditional', 'freeThrowsMade_traditional',
       'freeThrowsAttempted_traditional', 'reboundsOffensive_traditional',
       'reboundsDefensive_traditional',
       ...
       'OPP_percentageAssists_usage', 'OPP_percentageTurnovers_usage',
       'OPP_percentageSteals_usage', 'OPP_percentageBlocks_usage',
       'OPP_percentageBlocksAllowed_usage',
       'OPP_percentagePersonalFouls_usage',
       'OPP_percentagePersonalFoulsDrawn_usage', 'OPP_percentagePoints_usage',
       'OPP_plusMinusPoints_traditional', 'OPP_GAME_DATE'],
      dtype='object', length=167)

In [ ]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,GAME_DATE,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentage

# 🏠 Ajout IS_HOME et IS_WIN


In [ ]:
# Assurer le format datetime pour GAME_DATE
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])

# Merge avec les infos de match (MATCHUP, SEASON)
match_dataset = match_dataset.merge(
    df_games[['GAME_ID', 'TEAM_ID', 'MATCHUP', 'SEASON']],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)

print("match_dataset after merge")
display(match_dataset)


#display lines with nan values on matchup
print("Lines with NaN in MATCHUP:")
display(match_dataset[match_dataset['MATCHUP'].isna()])

# Définir si l'équipe joue à domicile
match_dataset['IS_HOME'] = match_dataset['MATCHUP'].str.contains('vs').astype(int)

# Calcul du résultat (win) et écart de points
match_dataset['IS_WIN'] = (match_dataset['points_traditional'] > match_dataset['OPP_points_traditional']).astype(int)
match_dataset['POINT_DIFF'] = match_dataset['points_traditional'] - match_dataset['OPP_points_traditional']

# Conversion GAME_DATE et tri
match_dataset['GAME_DATE'] = pd.to_datetime(match_dataset['GAME_DATE'])
match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)


match_dataset after merge


,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,GAME_DATE,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentage

Lines with NaN in MATCHUP:


,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,GAME_DATE,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentage

In [ ]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,GAME_DATE,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentage

# SPECIAL FOR ODDS : Keep only season past 2010-11


In [ ]:
# # get first line with season 2010-11 and cut all lines before
# first_season = match_dataset[match_dataset['SEASON'] == '2010-11'].index[0]
# match_dataset = match_dataset.iloc[first_season:].reset_index(drop=True)
# match_dataset

# Merge Odds with dataset

In [ ]:
all_odds_df = merge_odds_csv_files(DATA_ODDS_HISTORY_DIR)

match_dataset = match_odds_with_dataset(all_odds_df, match_dataset)



------------------ Nombre de lignes supprimées pour cotes manquantes: 38 ------------------


In [ ]:
match_dataset

,GAME_ID,TEAM_ID,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,points_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_TEAM_ID,GAME_DATE,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_percentagePoints3pt_scoring,OPP_percentage

# Reorder columns for visualisation

In [ ]:
cols_first = ['GAME_ID', 'GAME_DATE', 'TEAM_ID', 'OPP_TEAM_ID', 'IS_HOME', 'IS_WIN','POINT_DIFF', 'points_traditional','ODDS','OPP_ODDS']
other_cols = [col for col in match_dataset.columns if col not in cols_first]
match_dataset = match_dataset[cols_first + other_cols]
match_dataset

,GAME_ID,GAME_DATE,TEAM_ID,OPP_TEAM_ID,IS_HOME,IS_WIN,POINT_DIFF,points_traditional,ODDS,OPP_ODDS,fieldGoalsMade_traditional,fieldGoalsAttempted_traditional,threePointersMade_traditional,threePointersAttempted_traditional,freeThrowsMade_traditional,freeThrowsAttempted_traditional,reboundsOffensive_traditional,reboundsDefensive_traditional,reboundsTotal_traditional,assists_traditional,steals_traditional,blocks_traditional,turnovers_traditional,foulsPersonal_traditional,MINUTES_PLAYED,pointsOffTurnovers_misc,pointsSecondChance_misc,pointsFastBreak_misc,pointsPaint_misc,blocksAgainst_misc,offensiveRating_advanced,defensiveRating_advanced,netRating_advanced,assistPercentage_advanced,assistToTurnover_advanced,assistRatio_advanced,offensiveReboundPercentage_advanced,defensiveReboundPercentage_advanced,reboundPercentage_advanced,turnoverRatio_advanced,effectiveFieldGoalPercentage_advanced,trueShootingPercentage_advanced,usagePercentage_advanced,estimatedPace_advanced,pace_advanced,possessions_advanced,PIE_advanced,effectiveFieldGoalPercentage_fourfactors,freeThrowAttemptRate_fourfactors,teamTurnoverPercentage_fourfactors,oppEffectiveFieldGoalPercentage_fourfactors,oppFreeThrowAttemptRate_fourfactors,oppTeamTurnoverPercentage_fourfactors,oppOffensiveReboundPercentage_fourfactors,foulsDrawn_misc,percentageFieldGoalsAttempted2pt_scoring,percentageFieldGoalsAttempted3pt_scoring,percentagePoints2pt_scoring,percentagePointsMidrange2pt_scoring,percentagePoints3pt_scoring,percentagePointsFastBreak_scoring,percentagePointsFreeThrow_scoring,percentagePointsOffTurnovers_scoring,percentagePointsPaint_scoring,percentageAssisted2pt_scoring,percentageUnassisted2pt_scoring,percentageAssisted3pt_scoring,percentageUnassisted3pt_scoring,percentageAssistedFGM_scoring,percentageUnassistedFGM_scoring,threePointersPercentage_traditional,freeThrowsPercentage_traditional,percentageFieldGoalsMade_usage,percentageFieldGoalsAttempted_usage,percentageThreePointersMade_usage,percentageThreePointersAttempted_usage,percentageFreeThrowsMade_usage,percentageFreeThrowsAttempted_usage,percentageReboundsOffensive_usage,percentageReboundsDefensive_usage,percentageReboundsTotal_usage,percentageAssists_usage,percentageTurnovers_usage,percentageSteals_usage,percentageBlocks_usage,percentageBlocksAllowed_usage,percentagePersonalFouls_usage,percentagePersonalFoulsDrawn_usage,percentagePoints_usage,plusMinusPoints_traditional,OPP_fieldGoalsMade_traditional,OPP_fieldGoalsAttempted_traditional,OPP_threePointersMade_traditional,OPP_threePointersAttempted_traditional,OPP_freeThrowsMade_traditional,OPP_freeThrowsAttempted_traditional,OPP_reboundsOffensive_traditional,OPP_reboundsDefensive_traditional,OPP_reboundsTotal_traditional,OPP_assists_traditional,OPP_steals_traditional,OPP_blocks_traditional,OPP_turnovers_traditional,OPP_foulsPersonal_traditional,OPP_points_traditional,OPP_MINUTES_PLAYED,OPP_pointsOffTurnovers_misc,OPP_pointsSecondChance_misc,OPP_pointsFastBreak_misc,OPP_pointsPaint_misc,OPP_blocksAgainst_misc,OPP_offensiveRating_advanced,OPP_defensiveRating_advanced,OPP_netRating_advanced,OPP_assistPercentage_advanced,OPP_assistToTurnover_advanced,OPP_assistRatio_advanced,OPP_offensiveReboundPercentage_advanced,OPP_defensiveReboundPercentage_advanced,OPP_reboundPercentage_advanced,OPP_turnoverRatio_advanced,OPP_effectiveFieldGoalPercentage_advanced,OPP_trueShootingPercentage_advanced,OPP_usagePercentage_advanced,OPP_estimatedPace_advanced,OPP_pace_advanced,OPP_possessions_advanced,OPP_PIE_advanced,OPP_effectiveFieldGoalPercentage_fourfactors,OPP_freeThrowAttemptRate_fourfactors,OPP_teamTurnoverPercentage_fourfactors,OPP_oppEffectiveFieldGoalPercentage_fourfactors,OPP_oppFreeThrowAttemptRate_fourfactors,OPP_oppTeamTurnoverPercentage_fourfactors,OPP_oppOffensiveReboundPercentage_fourfactors,OPP_foulsDrawn_misc,OPP_percentageFieldGoalsAttempted2pt_scoring,OPP_percentageFieldGoalsAttempted3pt_scoring,OPP_percentagePoints2pt_scoring,OPP_percentagePointsMidrange2pt_scoring,OPP_pe

# 🚀 Features avancées


In [ ]:

# Ajouter les features avancées aux stats de match
#match_dataset = add_advanced_boxscore_features(match_dataset)

# MOVED TO CONFIG
# features_to_roll = cols_to_sum + cols_to_weighted_avg
# features_to_roll += [f"OPP_{col}" for col in cols_to_sum + cols_to_weighted_avg]

# Calcul des features glissantes shiftées
match_dataset = compute_rolling_features(match_dataset, "TEAM_ID", ["TEAM_ID", "GAME_DATE"], features_to_roll, N_LIST, method="ewm")


match_dataset = compute_winrates(match_dataset, "TEAM_ID", "IS_WIN", "IS_HOME", N_LIST)
match_dataset = compute_win_ratio(match_dataset, "TEAM_ID", "IS_WIN", N_LIST)

match_dataset["IS_WIN_SHIFTED"] = match_dataset.groupby("TEAM_ID")["IS_WIN"].shift(1).fillna(0).astype(int)
match_dataset["WIN_STREAK"] = match_dataset.groupby("TEAM_ID").apply(
    lambda x: compute_win_streak(x, "TEAM_ID", "IS_WIN_SHIFTED")).reset_index(level=0, drop=True)
match_dataset = compute_side_win_streak(match_dataset, win_shifted_col="IS_WIN_SHIFTED")


# Calcul des jours de repos pour l'équipe et l'adversaire
match_dataset["DAYS_SINCE_LAST_GAME"] = compute_rest_days(match_dataset, "GAME_DATE", "TEAM_ID")
match_dataset["OPP_DAYS_SINCE_LAST_GAME"] = compute_rest_days(match_dataset, "GAME_DATE", "OPP_TEAM_ID")

# Avantage de repos
match_dataset["REST_ADVANTAGE"] = (
    match_dataset["DAYS_SINCE_LAST_GAME"] - match_dataset["OPP_DAYS_SINCE_LAST_GAME"]
)


match_dataset = compute_rolling_rest_advantage(
    match_dataset, "TEAM_ID", "IS_HOME", "REST_ADVANTAGE", N_LIST
)

match_dataset = compute_home_away_pts(
    match_dataset, "TEAM_ID", "IS_HOME", "points_traditional", "OPP_points_traditional", N_LIST
)

# match_dataset = rename_pts_against_columns(match_dataset)

match_dataset = compute_h2h(match_dataset, N_LIST)
# match_dataset = compute_h2h_pts_margin(match_dataset, N_LIST)
match_dataset = compute_h2h_season(match_dataset)
match_dataset = compute_h2h_streak(match_dataset)

match_dataset = compute_elo(match_dataset)
match_dataset = compute_elo_season(match_dataset)


e:\Documents_\Dev\NBA_Predictor\src\feature_builder.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"ROLL_{col}_{window}"] = (
e:\Documents_\Dev\NBA_Predictor\src\feature_builder.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"ROLL_{col}_{window}"] = (
e:\Documents_\Dev\NBA_Predictor\src\feature_builder.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axi

In [ ]:
pd.options.display.max_columns = None
display(match_dataset)

GAME_ID  GAME_DATE     TEAM_ID OPP_TEAM_ID  IS_HOME  IS_WIN  \
0      0021000001 2010-10-26  1610612738  1610612748      1.0     1.0   
1      0021000001 2010-10-26  1610612748  1610612738      0.0     0.0   
2      0021000002 2010-10-26  1610612756  1610612757      0.0     0.0   
3      0021000002 2010-10-26  1610612757  1610612756      1.0     1.0   
4      0021000003 2010-10-26  1610612745  1610612747      0.0     0.0   
...           ...        ...         ...         ...      ...     ...   
38076  0042400315 2025-05-28  1610612760  1610612750      1.0     1.0   
38077  0042400305 2025-05-29  1610612752  1610612754      1.0     1.0   
38078  0042400305 2025-05-29  1610612754  1610612752      0.0     0.0   
38079  0042400306 2025-05-31  1610612752  1610612754      0.0     0.0   
38080  0042400306 2025-05-31  1610612754  1610612752      1.0     1.0   

       POINT_DIFF  points_traditional  ODDS  OPP_ODDS  \
0             8.0                88.0  1.88      1.70   
1            -8.0                80.0  1.70      1.88   
2           -14.0                92.0  3.30      1.22   
3            14.0               106.0  1.22      3.30   
4            -2.0               110.0  3.50      1.20   
...           ...                 ...   ...       ...   
38076        60.0               248.0  1.25      3.91   
38077        34.0               222.0  1.54      2.46   
38078       -34.0               188.0  2.46      1.54   
38079       -34.0               216.0  2.38      1.58   
38080        34.0               250.0  1.58      2.38   

       fieldGoalsMade_traditional  fieldGoalsAttempted_traditional  \
0                            32.0                             69.0   
1                            27.0                             74.0   
2                            36.0                             74.0   
3                            43.0                             93.0   
4                            38.0                             91.0   
...                           ...                              ...   
38076                        92.0                            176.0   
38077                        88.0                            178.0   
38078                        60.0                            148.0   
38079                        82.0                            172.0   
38080                        92.0                            170.0   

       threePointersMade_traditional  threePointersAttempted_traditional  \
0                                8.0                                16.0   
1                                8.0                                20.0   
2                                9.0                                19.0   
3                               10.0                                20.0   
4                                8.0                                20.0   
...                              ...                                 ...   
38076                           28.0                                70.0   
38077                           16.0                                58.0   
38078                           20.0                                60.0   
38079                           18.0                                64.0   
38080                           34.0                                66.0   

       freeThrowsMade_traditional  freeThrowsAttempted_traditional  \
0                            16.0                             25.0   
1                            18.0                             25.0   
2                            11.0                             16.0   
3                            10.0                             15.0   
4                            26.0                             28.0   
...                           ...                              ...   
38076                        36.0                             42.0   
38077                        30.0                             44.0   
38078                        48.0                             58

# 🧽 Nettoyage et sauvegarde


In [ ]:


final_date = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
raw_save_path = os.path.join(DATA_FINAL_DATASET_DIR, f'nba_features_final_{final_date}.csv')
clean_save_path = os.path.join(DATA_FINAL_CLEANED_DATASET_DIR, f'nba_features_cleaned_final_{final_date}.csv')

os.makedirs(DATA_FINAL_DATASET_DIR, exist_ok=True)
os.makedirs(DATA_FINAL_CLEANED_DATASET_DIR, exist_ok=True)

match_dataset.to_csv(raw_save_path, index=False)

final_cleaned = match_dataset.drop(columns=features_to_roll+COLS_MATCH_REAL, errors='ignore')
final_cleaned.to_csv(clean_save_path, index=False)

print(f"✅ Fichier brut : {raw_save_path}")
print(f"✅ Fichier clean : {clean_save_path}")

final_cleaned

✅ Fichier brut : data\final_dataset\nba_features_final_2025-06-06_19-50-30.csv
✅ Fichier clean : data\final_cleaned_dataset\nba_features_cleaned_final_2025-06-06_19-50-30.csv


GAME_ID  GAME_DATE     TEAM_ID OPP_TEAM_ID  IS_HOME  IS_WIN  \
0      0021000001 2010-10-26  1610612738  1610612748      1.0     1.0   
1      0021000001 2010-10-26  1610612748  1610612738      0.0     0.0   
2      0021000002 2010-10-26  1610612756  1610612757      0.0     0.0   
3      0021000002 2010-10-26  1610612757  1610612756      1.0     1.0   
4      0021000003 2010-10-26  1610612745  1610612747      0.0     0.0   
...           ...        ...         ...         ...      ...     ...   
38076  0042400315 2025-05-28  1610612760  1610612750      1.0     1.0   
38077  0042400305 2025-05-29  1610612752  1610612754      1.0     1.0   
38078  0042400305 2025-05-29  1610612754  1610612752      0.0     0.0   
38079  0042400306 2025-05-31  1610612752  1610612754      0.0     0.0   
38080  0042400306 2025-05-31  1610612754  1610612752      1.0     1.0   

       POINT_DIFF  ODDS  OPP_ODDS   SEASON  ROLL_fieldGoalsMade_traditional_3  \
0             8.0  1.88      1.70  2010-11                                NaN   
1            -8.0  1.70      1.88  2010-11                                NaN   
2           -14.0  3.30      1.22  2010-11                                NaN   
3            14.0  1.22      3.30  2010-11                                NaN   
4            -2.0  3.50      1.20  2010-11                                NaN   
...           ...   ...       ...      ...                                ...   
38076        60.0  1.25      3.91  2024-25                          87.374876   
38077        34.0  1.54      2.46  2024-25                          75.856303   
38078       -34.0  2.46      1.54  2024-25                          84.454392   
38079       -34.0  2.38      1.58  2024-25                          81.928151   
38080        34.0  1.58      2.38  2024-25                          72.227196   

       ROLL_fieldGoalsMade_traditional_5  ROLL_fieldGoalsMade_traditional_10  \
0                                    NaN                                 NaN   
1                                    NaN                                 NaN   
2                                    NaN                                 NaN   
3                                    NaN                                 NaN   
4                                    NaN                                 NaN   
...                                  ...                                 ...   
38076                          85.950884                           85.348006   
38077                          76.992537                           77.826531   
38078                          84.830707                           85.689047   
38079                          80.661691                           79.676252   
38080                          76.553805                           81.018311   

       ROLL_fieldGoalsMade_traditional_25  ROLL_fieldGoalsMade_traditional_50  \
0                                     NaN                                 NaN   
1                                     NaN                                 NaN   
2                                     NaN                                 NaN   
3                                     NaN                                 NaN   
4                                     NaN                                 NaN   
...                                   ...                                 ...   
38076                           86.858070                           88.131818   
38077                           79.539550                           81.736804   
38078                           86.669817                           87.235653   
38079                           80.190354                           81.982419   
38080                           84.618293                           86.167588   

       ROLL_fieldGoalsMade_traditional_100  \
0                                      NaN   
1                                      NaN   
2                                      NaN   
3                                      NaN

In [ ]:
final_cleaned.columns

Index(['GAME_ID', 'GAME_DATE', 'TEAM_ID', 'OPP_TEAM_ID', 'IS_HOME', 'IS_WIN',
       'POINT_DIFF', 'ODDS', 'OPP_ODDS', 'SEASON',
       ...
       'H2H_LAST_200_WINRATE', 'H2H_LAST_200_COUNT', 'H2H_SEASON_WINS',
       'H2H_SEASON_MATCHES', 'H2H_SEASON_WINRATE', 'H2H_WIN_STREAK', 'ELO_PRE',
       'OPP_ELO_PRE', 'ELO_PRE_SEASON', 'OPP_ELO_PRE_SEASON'],
      dtype='object', length=1242)

In [ ]:
#store end time of notebook
end_time = datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-06-06 19:52:56.379113
Total time:  0:04:54.229970
